# Notebook 2 — Descriptive Analysis & Validation

**Project:** Robson Classification and C-Section Predictive System
**Stage:** Phase 2 — Validating the classification, analysing patterns, comparing with Egypt
**Scripts covered:**

| Script | Role |
|--------|------|
| `robson_analysis.py` | Stage 2 analysis: external validation, Group 9 sensitivity, corrected VBAC, practice patterns, safety outcomes |
| `find_cs_var.py` | Exploratory: finding the caesarean variable in Egypt HIS 2015 data |

**9 CSV outputs documented:**

| File | Contents |
|------|----------|
| `01_cdc_2023_robson_report.csv` | WHO standard report table — CDC 2023 |
| `02_egypt_benha_2018_robson_report.csv` | WHO report table — Benha University Hospital |
| `03_egypt_qena_2022_robson_report.csv` | WHO report table — Qena (South Valley) |
| `04_three_way_comparison.csv` | Side-by-side: CDC vs Benha vs Qena |
| `05_indirect_standardisation.csv` | Expected vs observed CS rates (⚠ old 10-category — see note) |
| `06_cdc_safety_outcomes_by_group.csv` | Apgar, NICU, near-miss by Robson group |
| `07_group1_payer_standardised.csv` | Group 1 CS rate by payer (age-standardised) |
| `08_group1_time_of_day_standardised.csv` | Group 1 CS rate by time of day |
| `09_egypt_national_cs_rates.csv` | Egypt national CS rates timeline (2005–2026) |

---

### What changed from Notebook 1

Notebook 1 built the classifier and ran it. This notebook **validates** the output against independently published statistics, investigates known problems (Group 9, VBAC logic), and produces the comparison tables that feed into the Egypt analysis.

### ⚠ Important note on `05_indirect_standardisation.csv`

This file contains the **old 10-category** standardisation results (Benha gap = +7.54pp). These were later corrected to the **8-category** method in `egypt_gap_v2.py` (Notebook 3), which produces the final numbers:

| | Old (10-cat) | **Corrected (8-cat)** |
|---|---|---|
| Benha Gap A | +7.54 pp | **+4.46 pp** |
| Qena Gap A | +20.69 pp | **+20.65 pp** |

The correction happened because onset-of-labour groups (1 vs 2, 3 vs 4) were merged — the split reflects clinical decisions, not case-mix. See Notebook 3 for the full explanation.

---

## Part 1 — `robson_analysis.py`: What changed from the first pipeline, and why

This script is Stage 2 — it takes the classified data from Notebook 1 and performs five analyses. But first, it documents **five corrections** over the first pipeline:

1. **Caching** — Parse the 3.8 GB file once, save as Parquet, load in seconds afterwards
2. **Real validation targets** — Replace generic global benchmarks with published US statistics for the same year
3. **Group 9 sensitivity** — Quantify the effect of CDC's mixed "other presentation" category
4. **VBAC logic corrected** — The first script had identical TOLAC and VBAC numbers (impossible)
5. **Practice patterns restricted to Group 1** — Remove the attendant and weekday artefacts

### 1.1 — Imports and Caching

The script imports directly from `robson_classifier.py` and `cdc_robson_pipeline.py` — the functions we built in Notebook 1. The Parquet cache means the 10-minute load happens only once.

In [ ]:
"""
=============================================================================
ROBSON ANALYSIS — STAGE 2
Validation against published national statistics + corrected VBAC logic
=============================================================================

WHAT CHANGED FROM cdc_robson_pipeline.py, AND WHY

1. CACHING. The raw file is parsed once and saved as Parquet. Subsequent runs
   load in seconds instead of ~10 minutes. (Supervisor's technical note:
   "save a filtered version once rather than reloading the raw file.")

2. REAL VALIDATION TARGETS. The benchmark ranges in the first script were
   generic global figures and were wrong for the United States. They flagged
   correct output as suspect. They are replaced here with published US
   statistics for the same data year, which is a genuine external check:
     - Overall caesarean rate, United States 2023 .......... 32.3%
     - NTSV (low-risk first birth) caesarean rate .......... ~26%
     - Labour induction rate, all births ................... ~33-34%
   NTSV is exactly Robson groups 1 + 2 combined, and it is published
   independently by NCHS/ACOG. If our classifier reproduces it, the
   nulliparous branch of the rule tree is correct.

3. GROUP 9 SENSITIVITY. CDC records presentation as cephalic / breech /
   "other". WHO group 9 is strictly transverse or oblique lie and is ~100%
   caesarean by definition. Our group 9 sits near 46%, which means "other"
   is a mixed category that also contains face, brow and compound
   presentations — some of which deliver vaginally. We do not silently
   reassign them. We report the group as-is, label it honestly, and run a
   sensitivity analysis moving them to unclassifiable so the effect on every
   other group is visible and auditable.

4. VBAC LOGIC CORRECTED. The first script reported identical numbers for
   "TOLAC attempted" and "VBAC successes" — impossible. A failed trial of
   labour ends in a caesarean and is coded as a repeat caesarean,
   indistinguishable from a planned repeat caesarean without ME_TRIAL.
   Corrected:  TOLAC attempted = VBAC + (repeat caesarean AND ME_TRIAL = Y)

5. PRACTICE-PATTERN ANALYSIS RESTRICTED TO GROUP 1. Two artefacts made the
   unrestricted version misleading; both are explained inline below.
"""

import pandas as pd
import numpy as np
from pathlib import Path
import sys

sys.path.insert(0, str(Path(__file__).parent))
from robson_classifier import classify, report_table
from cdc_robson_pipeline import (
    load_puf, add_is_cesarean, add_practice_vars, add_near_miss_flag
)

CACHE = "natality_2023_filtered.parquet"


# =============================================================================
# LOAD (with cache)
# =============================================================================

def get_data(path, rebuild=False):
    if Path(CACHE).exists() and not rebuild:
        print(f"Loading cached {CACHE} ...")
        df = pd.read_parquet(CACHE)
        print(f"Loaded {len(df):,} records from cache.\n")
        return df

    df = load_puf(path)
    df = add_is_cesarean(df)
    df = add_practice_vars(df)
    df = add_near_miss_flag(df)
    df = classify(df, subgroups=True)

### 1.2 — External Validation: The Real Stop-Gate

This is a **far stronger check** than comparing group sizes against generic ranges. These figures were produced by NCHS from the same birth certificates using their own code. If our classifier reproduces them, the implementation is correct.

| Metric | Our result | Published | Source |
|--------|-----------|-----------|--------|
| Overall CS rate | 32.35% | 32.3% | NCHS, Births: Final Data for 2023 |
| NTSV CS rate (Groups 1+2) | 26.56% | ~25.6–26.6% | ACOG 2025 / NCHS |
| Groups 2b, 4b CS rate | 100% | 100% | By definition |

**Why NTSV matters:** NTSV is exactly Robson groups 1 + 2 combined, and it is published independently by NCHS/ACOG. If our classifier reproduces it, the **nulliparous branch** of the rule tree is correct — this is the branch where the onset-of-labour derivation matters most.

In [ ]:
# =============================================================================
# EXTERNAL VALIDATION — the real stop gate
# =============================================================================

def external_validation(df):
    """
    Compare our derived figures against independently published US statistics
    for 2023. This is a far stronger check than comparing group sizes against
    generic international ranges, because these figures were produced by NCHS
    from the same underlying birth certificates using their own code.
    """
    print("=" * 74)
    print("EXTERNAL VALIDATION — derived figures vs. published US statistics")
    print("=" * 74)

    d = df[df["Classification_status"] == "Classified"]
    rows = []

    # --- 1. Overall caesarean rate -----------------------------------------
    ours = 100 * d["is_cesarean"].mean()
    rows.append(("Overall caesarean rate", ours, "32.3%",
                 "NCHS, Births: Final Data for 2023"))

    # --- 2. NTSV = Robson groups 1 + 2a + 2b --------------------------------
    ntsv = d[d["Robson_group"].isin(["1", "2a", "2b"])]
    ntsv_rate = 100 * ntsv["is_cesarean"].mean()
    rows.append(("NTSV caesarean rate (groups 1+2)", ntsv_rate, "~25.6-26.6%",
                 "ACOG 2025 / NCHS low-risk cesarean"))

    # --- 3. Induction rate among term singleton cephalic, no previous CS ----
    #     (groups 1-4 denominator; groups 2a + 4a numerator)
    g1234 = d[d["Robson_group"].isin(["1", "2a", "2b", "3", "4a", "4b"])]
    induced = d[d["Robson_group"].isin(["2a", "4a"])]
    ind_rate = 100 * len(induced) / len(g1234)
    rows.append(("Induction rate (groups 1-4 denominator)", ind_rate, "~40%",
                 "NCHS: 34.5% of ALL births 2024; higher at term"))

    # --- 4. Structural checks: groups that must be 100% by definition -------
    for grp in ["2b", "4b"]:
        sub = d[d["Robson_group"] == grp]

### 1.3 — Group 9 Sensitivity Analysis

**The problem:** WHO Group 9 = singleton, transverse or oblique lie, ~100% CS by definition. But CDC codes presentation as cephalic / breech / "other" — and "other" is a catch-all that includes face, brow, and compound presentations (some of which deliver vaginally).

**Our Group 9 CS rate ≈ 46%** — proof that it contains non-transverse presentations.

**The decision:** Do NOT silently reassign. Report as-is, label honestly, and run sensitivity analysis showing the effect on all other groups is negligible.

**Why this matters for Egypt:** Group 9 gets a reference rate of 100% in the indirect standardisation. This is correct by medical definition (a transverse lie cannot deliver vaginally), not because of the CDC data.

In [ ]:
for name, ours_v, pub, src in rows:
        print(f"{name:<42} {ours_v:>8.2f}%   {pub:>16}   {src}")

    # --- Group 5 internal consistency --------------------------------------
    print("\nInternal consistency — Robson group 5 (previous caesarean):")
    g5 = d[d["Robson_group"].isin(["5", "5.1", "5.2"])]
    print(f"  Group 5 as % of all births      : {100*len(g5)/len(d):>6.2f}%")
    print(f"  Caesarean rate within group 5   : {100*g5['is_cesarean'].mean():>6.2f}%")
    print("  Expected: group 5 is the largest single contributor to the")
    print("  overall rate in almost every published Robson study.")
    print()


# =============================================================================
# GROUP 9 SENSITIVITY
# =============================================================================

def group9_sensitivity(df):
    """
    WHO group 9 = singleton, transverse or oblique lie, ~100% caesarean.
    CDC 'other' presentation is broader than that. Rather than silently
    reassigning these records, quantify the effect of excluding them.
    """
    print("=" * 74)
    print("GROUP 9 SENSITIVITY — CDC 'other' presentation is not WHO group 9")
    print("=" * 74)

    d = df[df["Classification_status"] == "Classified"].copy()
    g9 = d[d["Robson_group"] == "9"]

    print(f"  Records assigned to group 9      : {len(g9):>10,}  ({100*len(g9)/len(d):.2f}% of births)")
    print(f"  Caesarean rate within group 9    : {100*g9['is_cesarean'].mean():>10.2f}%")
    print(f"  Expected if strictly transverse  : {'~100%':>10}")

### 1.4 — VBAC / TOLAC Corrected

**The bug in the first script:** It reported identical numbers for "TOLAC attempted" and "VBAC successes" — which is impossible. A failed trial of labour ends in a caesarean and is coded as a repeat caesarean (RDMETH_REC = 4), indistinguishable from a planned repeat caesarean without `ME_TRIAL`.

**The correction:**
- TOLAC attempted = VBAC successes + (repeat CS where ME_TRIAL = "Y")
- These are **two distinct numbers**, and the distinction is clinically important:
  - **TOLAC rate** = how often a trial of labour is offered/attempted
  - **VBAC success rate** = how often an attempt succeeds
  - **Egypt's problem is the first number**, not the second

**Relevance:** This analysis stays descriptive — no predictive VBAC model. The supervisor deferred it to Phase 2, and the technical reason (selection bias in who gets offered TOLAC) supports that decision.

In [ ]:
print("  that CDC folds into the same 'other' code. This is a limitation")
    print("  of birth certificate data, not an error in the rule tree.")
    print()

    # Effect of removing group 9 entirely
    without = d[d["Robson_group"] != "9"]
    print(f"  Overall caesarean rate WITH group 9    : {100*d['is_cesarean'].mean():.3f}%")
    print(f"  Overall caesarean rate WITHOUT group 9 : {100*without['is_cesarean'].mean():.3f}%")
    print(f"  Absolute difference                    : "
          f"{abs(100*d['is_cesarean'].mean() - 100*without['is_cesarean'].mean()):.3f} pp")
    print()
    print("  Conclusion: the effect on every other group is negligible.")
    print("  Recommended handling — report group 9 as computed, labelled")
    print("  'CDC other presentation (not strictly transverse/oblique)',")
    print("  and state the limitation. Do not reassign.")
    print()


# =============================================================================
# CORRECTED VBAC / TOLAC
# =============================================================================

def vbac_corrected(df):
    """
    RDMETH_REC (delivery method, expanded):
        1 = Vaginal (no previous caesarean)
        2 = VBAC — vaginal birth after previous caesarean
        3 = Primary caesarean
        4 = Repeat caesarean
    ME_TRIAL = trial of labour attempted (recorded for caesareans only)

    A FAILED trial of labour is coded 4, exactly like a planned repeat
    caesarean. ME_TRIAL is the only field that separates them.
    """
    print("=" * 74)
    print("VBAC / TOLAC — CORRECTED")
    print("=" * 74)

    prev = df["RF_CESAR"].astype(str).str.strip() == "Y"
    rdm = df["RDMETH_REC"].astype(str).str.strip()
    trial = df["ME_TRIAL"].astype(str).str.strip()

    d = df[prev & rdm.isin(["2", "4"])].copy()
    d["vbac"] = rdm[prev & rdm.isin(["2", "4"])] == "2"
    d["repeat_cs"] = rdm[prev & rdm.isin(["2", "4"])] == "4"
    d["trial_y"] = trial[prev & rdm.isin(["2", "4"])] == "Y"

    # TOLAC attempted = achieved VBAC, OR repeat caesarean after a trial
    d["tolac"] = d["vbac"] | (d["repeat_cs"] & d["trial_y"])

    n = len(d)
    n_tolac = int(d["tolac"].sum())

### 1.5 — Practice Patterns: Why Restrict to Group 1?

**Two artefacts in the unrestricted analysis:**

**Attendant artefact:** Midwives showed CS rate ~0.9%, physicians ~37%. This is almost circular — "attendant at birth" records who *performed* the delivery, and a caesarean is performed by a physician by definition. A midwife-led labour that ends in CS is recorded with a physician attendant. The field cannot answer "do midwives have lower caesarean rates."

**Weekday artefact:** Weekday 34.6% vs weekend 24.2% is scheduling — planned pre-labour caesareans are booked on working days. It's not evidence of a decision-making difference.

**Group 1 removes both problems:** Nulliparous, term, single, cephalic, spontaneous labour. Every woman arrived in labour on her own. No caesarean was scheduled. Any remaining variation is variation in how **labour was managed** — which is the cleanest comparator for Egypt.

In [ ]:
print(f"    → succeeded (VBAC)                  : {n_vbac:>10,}   ({100*n_vbac/n_tolac:.1f}% of attempts)")
    print(f"    → failed (ended in caesarean)       : {n_failed:>10,}   ({100*n_failed/n_tolac:.1f}% of attempts)")
    print(f"  VBAC rate (of all women with prev CS) : {100*n_vbac/n:>9.1f}%")
    print()
    print("  Two distinct numbers, and the distinction is the whole point:")
    print("    TOLAC rate    = how often a trial of labour is OFFERED / attempted")
    print("    VBAC success  = how often an attempt SUCCEEDS")
    print("  Egypt's problem is the first number, not the second.")
    print()

    for var in ["payer", "attendant"]:
        t = d.groupby(var).agg(
            n=("tolac", "size"),
            tolac_rate=("tolac", "mean"),
        )
        t["success_rate"] = d[d["tolac"]].groupby(var)["vbac"].mean()
        t["tolac_rate"] = (100 * t["tolac_rate"]).round(1)
        t["success_rate"] = (100 * t["success_rate"]).round(1)
        print(f"  By {var}:")
        print(t.to_string())
        print()
    return d


# =============================================================================
# PRACTICE PATTERNS — restricted, with the artefacts explained
# =============================================================================

def practice_patterns(df):
    print("=" * 74)
    print("PRACTICE-PATTERN ANALYSIS — restricted to Robson group 1")
    print("=" * 74)
    print("""
  Why restrict to group 1?

  ATTENDANT ARTEFACT. In the unrestricted analysis, midwives showed a
  caesarean rate near 0.9% and physicians near 37%. That comparison is

### 1.6 — Safety Outcomes by Robson Group

**The argument:** If a group with a much higher caesarean rate does NOT show correspondingly better neonatal outcomes, the extra caesareans are not buying safety.

This uses:
- **Apgar < 7 at 5 minutes** — neonatal distress indicator
- **NICU admission** — severity indicator
- **Maternal near-miss** — the WHO proxy from Notebook 1

**Key comparison:** Groups 1 and 3 are both spontaneous labour, term, singleton, cephalic — the difference is parity (1 = first birth, 3 = subsequent). Reading them together shows whether the clinical profile, not the CS rate, drives outcomes.

In [ ]:
WEEKDAY ARTEFACT. Weekday 34.6% vs weekend 24.2% is largely scheduling:
  planned pre-labour caesareans are booked on working days. It is not
  evidence of a decision-making difference.

  Group 1 removes both problems — nulliparous, term, single, cephalic,
  SPONTANEOUS labour. Every woman in it arrived in labour on her own, and
  no caesarean in it was scheduled. Any variation left is variation in how
  labour was managed. This is also the cleanest comparator for Egypt.
""")

    d = df[(df["Classification_status"] == "Classified") &
           (df["Robson_group"] == "1")]

    print(f"  Group 1 size: {len(d):,} births, caesarean rate "
          f"{100*d['is_cesarean'].mean():.2f}%\n")

    for var in ["payer", "weekend", "time_of_day"]:
        t = d.groupby(var, observed=True)["is_cesarean"].agg(["mean", "size"])
        t["cs_rate_%"] = (100 * t["mean"]).round(2)
        print(f"  Group 1 caesarean rate by {var}:")
        print(t[["cs_rate_%", "size"]].to_string())
        print()


# =============================================================================
# SAFETY OUTCOMES BY GROUP
# =============================================================================

def safety_by_group(df):
    """
    Does a higher caesarean rate in a group buy better outcomes?

### 1.7 — The Main Execution Block

Runs all analyses in sequence and saves the final report table.

In [ ]:
print("=" * 74)

    d = df[df["Classification_status"] == "Classified"].copy()
    d["apgar5_low"] = pd.to_numeric(d["APGAR5"], errors="coerce").lt(7)
    d["nicu"] = d["AB_NICU"].astype(str).str.strip() == "Y"

    t = d.groupby(["Robson_group_sort_order", "Robson_group"], observed=True).agg(
        n=("is_cesarean", "size"),
        cs_rate=("is_cesarean", "mean"),
        apgar5_lt7=("apgar5_low", "mean"),
        nicu_admit=("nicu", "mean"),
        near_miss=("near_miss_flag", "mean"),
    ).reset_index().sort_values("Robson_group_sort_order")

    for c in ["cs_rate", "apgar5_lt7", "nicu_admit", "near_miss"]:
        t[c] = (100 * t[c]).round(2)
    print(t.drop(columns="Robson_group_sort_order").to_string(index=False))
    print()
    print("  Read groups 1 and 3 together: both are spontaneous labour, term,")
    print("  singleton, cephalic. If a group with a much higher caesarean rate")
    print("  does not show correspondingly better neonatal outcomes, the extra")
    print("  caesareans are not buying safety. That is the argument, made on")
    print("  real data, before Egypt is mentioned at all.")
    print()


# =============================================================================
if __name__ == "__main__":
    import argparse
    p = argparse.ArgumentParser()
    p.add_argument("path", nargs="?", default="Nat2023PublicUS.txt")
    p.add_argument("--rebuild", action="store_true", help="Ignore the Parquet cache")
    args = p.parse_args()

    df = get_data(args.path, rebuild=args.rebuild)

    print()
    print("=" * 74)
    print("WHO STANDARD ROBSON REPORT TABLE — CDC Natality 2023")
    print("=" * 74)
    rt = report_table(df[df["Classification_status"] == "Classified"])
    print(rt.to_string(index=False))
    rt.to_csv("robson_report_2023_final.csv", index=False)
    print("\nSaved: robson_report_2023_final.csv\n")

    external_validation(df)
    group9_sensitivity(df)
    vbac_corrected(df)
    practice_patterns(df)
    safety_by_group(df)

    print("=" * 74)
    print("STAGE 2 COMPLETE.")
    print("=" * 74)

---

## Part 2 — `find_cs_var.py`: Exploratory Script

**Purpose:** Find the caesarean delivery variable in the Egypt Health Issues Survey 2015 (EHIS) Stata file (`EGIQ73FL.DTA`).

This is a one-time exploratory script — it reads the variable labels from the Stata file and searches for keywords related to delivery. It was used early in the project to identify the correct field before building `ehis_2015.py` (Notebook 3).

**What it found:** The key variables for delivery method in EHIS 2015, which led to the discovery that **EHIS 43.8% is NOT a caesarean rate** — it's the share of *mothers* who ever had a caesarean, not the share of *births* that were caesarean. A fundamentally different denominator.

In [ ]:
import pandas as pd

it = pd.read_stata("EGIQ73FL.DTA", convert_categoricals=False, iterator=True)
labels = it.variable_labels()

keys = ["cesarean", "caesarean", "c-section", "delivery", "deliver",
        "birth", "born", "pregnan"]

print(f"{len(labels)} variables total\n")
for var, lab in labels.items():
    if any(k in str(lab).lower() for k in keys):
        print(f"{var:<12} {lab}")

---

## Part 3 — The 9 CSV Output Files

These files are the **tangible outputs** of the analysis pipeline. Every number in the dashboard and the final report traces back to one of these files.

### 3.1 — `01_cdc_2023_robson_report.csv`: The Reference

The WHO standard report table on 3.6 million US births. This is the **reference** against which Egypt is compared.

| Column | Meaning |
|--------|---------|
| `robson_group` | 1, 2a, 2b, 3, 4a, 4b, 5.1, 5.2, 6, 7, 8, 9, 10 |
| `n_women` | Number of births in the group |
| `n_cesarean` | Caesareans within the group |
| `group_size_pct` | Group as % of all births |
| `group_cs_rate_pct` | CS rate within the group |
| `absolute_contribution_pct` | Group's CS / all births |
| `relative_contribution_pct` | Group's share of all CS |

In [ ]:
import pandas as pd

cdc = pd.read_csv("/mnt/user-data/uploads/01_cdc_2023_robson_report.csv")
print("CDC 2023 Robson Report Table")
print(f"Total births: {cdc['n_women'].sum():,}")
print(f"Total CS: {cdc['n_cesarean'].sum():,}")
print(f"Overall CS rate: {100 * cdc['n_cesarean'].sum() / cdc['n_women'].sum():.2f}%")
print()
print(cdc.to_string(index=False))

### 3.2 — Egyptian Hospital Reports: Benha and Qena

Same structure as the CDC report, but for two Egyptian hospitals. These are the only two published Egyptian studies with complete Robson group distributions.

**Key differences from CDC:**
- Benha uses 10 groups (no subgroups) — so "Group 2" includes both 2a and 2b
- Qena uses 10 groups similarly
- Group 5 is dramatically larger in both (Benha 36.2%, Qena 40.8% vs CDC 12.5%)

In [ ]:
benha = pd.read_csv("/mnt/user-data/uploads/02_egypt_benha_2018_robson_report.csv")
qena = pd.read_csv("/mnt/user-data/uploads/03_egypt_qena_2022_robson_report.csv")

print("=== BENHA 2018 (n=850) ===")
print(f"Overall CS rate: {100 * benha['n_cesarean'].sum() / benha['n_women'].sum():.2f}%")
print(benha.to_string(index=False))
print()
print("=== QENA 2022-23 (n=2,186) ===")
print(f"Overall CS rate: {100 * qena['n_cesarean'].sum() / qena['n_women'].sum():.2f}%")
print(qena.to_string(index=False))

### 3.3 — `04_three_way_comparison.csv`: The Core Comparison

Side-by-side comparison of group sizes and CS rates across CDC, Benha, and Qena. The `*_minus_cdc_pp` columns show the difference in percentage points.

**This table tells the story:** Where Egypt differs from the US, and by how much.

In [ ]:
comp = pd.read_csv("/mnt/user-data/uploads/04_three_way_comparison__1_.csv")
print("Three-Way Comparison: CDC vs Benha vs Qena")
print(comp.to_string(index=False))

### 3.4 — `05_indirect_standardisation.csv`: Expected vs Observed

⚠ **These are the OLD 10-category numbers.** The corrected 8-category results (Notebook 3) are:

| Hospital | Gap A (old 10-cat) | **Gap A (corrected 8-cat)** |
|----------|-------------------|-----------------------------|
| Benha | +7.54 pp | **+4.46 pp** |
| Qena | +20.69 pp | **+20.65 pp** |

The file is preserved as a record of the analytical evolution. The corrected file is `egypt_case_mix_8cat.csv` produced by `egypt_gap_v2.py`.

In [ ]:
std = pd.read_csv("/mnt/user-data/uploads/05_indirect_standardisation.csv")
print("Indirect Standardisation (OLD 10-category — see note above)")
print(std.to_string(index=False))

### 3.5 — `06_cdc_safety_outcomes_by_group.csv`: Does More CS = Better Outcomes?

Safety indicators (Apgar <7, NICU, near-miss) by Robson group in the CDC reference population.

**The argument this table makes:** Compare Groups 1 and 3 — both spontaneous labour, term, singleton, cephalic. Group 1 (nulliparous) has a higher CS rate than Group 3 (multiparous), but does it show better neonatal outcomes? If not, the extra caesareans are not buying safety.

In [ ]:
safety = pd.read_csv("/mnt/user-data/uploads/06_cdc_safety_outcomes_by_group.csv")
print("Safety Outcomes by Robson Group (CDC 2023)")
print(safety.to_string(index=False))

### 3.6 — Group 1 Practice Patterns (Payer and Time-of-Day)

These two files show CS rate variation within Group 1 — where every woman arrived in spontaneous labour and no CS was scheduled.

**`07_group1_payer_standardised.csv`** — age-standardised rates by payer. After standardisation, Medicaid and Private converge (13.7% vs 13.1%), meaning the crude difference was driven by age, not by financial incentive. Self-pay remains lower (9.0%).

**`08_group1_time_of_day_standardised.csv`** — CS rate by time of day. If rates spike at shift-change hours, it suggests staffing pressure, not clinical need.

In [ ]:
payer = pd.read_csv("/mnt/user-data/uploads/07_group1_payer_standardised.csv")
print("Group 1 CS Rate by Payer (age-standardised)")
print(payer.to_string(index=False))
print()

tod = pd.read_csv("/mnt/user-data/uploads/08_group1_time_of_day_standardised.csv")
print("Group 1 CS Rate by Time of Day")
print(tod.to_string(index=False))

### 3.7 — `09_egypt_national_cs_rates.csv`: The Egyptian Trajectory

National CS rates from 2005 to 2026. This is the context that frames the entire project.

**Critical data quality notes embedded in this file:**
- **2015 (43.8%)** is NOT a CS rate — it's the share of *mothers* who ever had a CS (denominator = women, not births). Do NOT plot on the same trend line as the other rows.
- **2014 (51.8%)** is the real CS rate from DHS microdata (denominator = births)
- **2025 (80%)** from MoH electronic registration — 93% of these in the private sector
- **2026 Q1 (63%)** from the Deputy Minister — Q1 only, verified April 2026

This table resolves the "43.8% vs 51.8%" contradiction that confused early analysis: they measure different things.

In [ ]:
egypt = pd.read_csv("/mnt/user-data/uploads/09_egypt_national_cs_rates.csv")
print("Egypt National CS Rates (2005-2026)")
for _, row in egypt.iterrows():
    note = f"  ⚠ {row['note']}" if pd.notna(row['note']) and len(str(row['note'])) > 5 else ""
    print(f"  {int(row['year'])}  {row['cs_rate_pct']:>5.1f}%  {row['source_type']:<20}  {row['source'][:60]}")
    if note:
        print(f"        {note[:100]}")

---

## Summary

### Validations passed
| Check | Result |
|-------|--------|
| Overall CS rate | 32.35% vs 32.3% published ✓ |
| NTSV rate | 26.56% vs ~26% published ✓ |
| Groups 2b, 4b | 100% CS ✓ |
| Group 9 | Documented as mixed, sensitivity shows negligible effect ✓ |
| VBAC logic | Corrected: TOLAC ≠ VBAC (distinct numbers) ✓ |

### Corrections documented
| Issue | What was wrong | What was fixed |
|-------|---------------|----------------|
| VBAC = TOLAC | First script counted only VBAC, missed failed trials | TOLAC = VBAC + (repeat CS with ME_TRIAL=Y) |
| Attendant artefact | Midwives ~0.9% vs physicians ~37% looked clinical | It's circular — CS is performed by physicians by definition |
| Weekday artefact | Weekday 34.6% vs weekend 24.2% | Scheduling, not decision-making. Restrict to Group 1 |
| Group 9 | CS rate ~46% but "should be ~100%" | CDC "other" ≠ WHO transverse. Report and label, don't reassign |
| 10-category standardisation | Benha gap inflated (+7.54) | Corrected to 8-category (+4.46) in Notebook 3 |

### Next notebooks

- **Notebook 3** → Egyptian data sources & corrected indirect standardisation (8-category)
- **Notebook 4** → Predictive model (XGBoost)
- **Notebook 5** → SHAP interpretation & temporal validation
- **Notebook 6** → Export & packaging

---
*End of Notebook 2*